# Joint representation analysis and cached t-SNE
Reorganized only; no experiment cells executed during creation. Existing results were copied with byte-hash verification.

In [ ]:
def run_representation():
    # CELL 02 — Imports, paths and reproducibility
    from pathlib import Path
    import gc
    import random
    import hashlib
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import matplotlib.pyplot as plt
    from torchvision import datasets, models, transforms
    from torchvision.transforms import InterpolationMode
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm
    from IPython.display import display
    import open_clip

    DATA_DIR = TASK_DIR / 'data'
    BASELINE_PATH = TASK_DIR / 'results' / 'clean_baseline.pt'
    OUTPUT_DIR = TASK_DIR / 'results' / 'representation_analysis'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    SEED = 6304
    BATCH_SIZE = 32
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    print('Device:', DEVICE)
    print('Results:', OUTPUT_DIR)

    from sklearn.metrics import f1_score

    import json
    from torchvision.transforms import functional as TF
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from matplotlib.lines import Line2D

    # CELL 03 — Load baseline, clean images and saved test IDs
    baseline = torch.load(BASELINE_PATH, map_location='cpu', weights_only=True)
    assert baseline['seed'] == SEED
    class_names = list(baseline['class_names'])
    num_classes = len(class_names)
    test_indices = baseline['test_indices'].long().numpy()
    official_test = datasets.STL10(str(DATA_DIR), split='test', download=False)
    assert list(official_test.classes) == class_names
    y_test = torch.tensor(np.asarray(official_test.labels)[test_indices], dtype=torch.long)
    assert len(test_indices) == 500 and len(np.unique(test_indices)) == 500

    common_preprocess = transforms.Compose([
        transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.ToTensor(),
    ])
    # Approximately 300 MB, shared across all models. No transformed copies retained.
    clean_images = torch.stack([
        common_preprocess(official_test[int(i)][0].convert('RGB')) for i in test_indices
    ])
    heads = {}
    for name, state in baseline['head_states'].items():
        out_dim, in_dim = state['weight'].shape
        head = nn.Linear(in_dim, out_dim)
        head.load_state_dict(state)
        heads[name] = head.eval().requires_grad_(False)
    DISPLAY_NAMES = {
        'resnet50': 'ResNet-50 + linear head',
        'vit_b16': 'ViT-B/16 + linear head',
        'clip': 'CLIP ViT-B/32 + linear head',
    }
    ZERO_SHOT_NAME = 'CLIP ViT-B/32 zero-shot'
    clean_logits = baseline['clean_logits']
    clean_predictions = {name: logits.argmax(1) for name, logits in clean_logits.items()}
    for logits in clean_logits.values(): assert logits.shape == (500, num_classes)
    manifest = pd.DataFrame({'official_test_index': test_indices, 'label': y_test.numpy()})
    manifest['image_id'] = [f'stl10/test/{i:05d}' for i in test_indices]
    manifest['class_name'] = [class_names[int(i)] for i in y_test]
    print('Loaded 500 images and all trained heads. No training required.')

    baseline_hash = hashlib.sha256(BASELINE_PATH.read_bytes()).hexdigest()
    clean_features = baseline['clean_test_features']
    id_to_position = {int(image_id): position for position, image_id in enumerate(test_indices)}
    cue = load_preserved_conflicts()[0]
    cue_positions = torch.tensor([id_to_position[int(i)] for i in cue['content_indices']], dtype=torch.long)
    assert torch.equal(y_test[cue_positions], cue['content_targets'])
    assert len(cue['content_indices']) >= 200
    for name in DISPLAY_NAMES:
        assert clean_features[name].shape[0] == len(test_indices)
        assert cue['features'][name].shape[0] == len(cue_positions)
    print('Cue conflicts matched to their clean content images by ID.')

    @torch.no_grad()
    def encode_images(backbone, normalize, images, name):
        parts = []
        loader = DataLoader(images, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        for batch in tqdm(loader, desc=name):
            batch = normalize(batch.to(DEVICE))
            if name == 'clip': features = F.normalize(backbone.encode_image(batch).float(), dim=-1)
            else: features = backbone(batch).float()
            parts.append(features.cpu())
        return torch.cat(parts)

    @torch.no_grad()
    def clip_text_setup(backbone):
        tokens = open_clip.get_tokenizer('ViT-B-32')([
            f'a photo of a {name}.' for name in class_names
        ]).to(DEVICE)
        return (F.normalize(backbone.encode_text(tokens).float(), dim=-1).cpu(),
                backbone.logit_scale.exp().float().cpu())


    def atomic_save(package, path):
        temporary = path.with_suffix('.tmp')
        torch.save(package, temporary)
        temporary.replace(path)

    def verify_regular(package):
        assert package['baseline_sha256'] == baseline_hash, 'Baseline checkpoint changed.'
        assert torch.equal(package['test_indices'], torch.as_tensor(test_indices)), 'Image order differs.'
        assert package['features'].shape[0] == len(test_indices)
        return package

    # CELL 04 — PREVIEW missing grayscale intervention before extraction
    fig, axes = plt.subplots(3, 2, figsize=(7, 9))
    for row, position in enumerate([0, 1, 2]):
        original = clean_images[position]
        gray = TF.rgb_to_grayscale(original, num_output_channels=3)
        for col, (image, title) in enumerate([(original, 'Clean'), (gray, 'Grayscale')]):
            axes[row, col].imshow(image.permute(1, 2, 0).numpy())
            axes[row, col].set_title(f'{title}: {class_names[int(y_test[position])]}')
            axes[row, col].axis('off')
    plt.tight_layout()
    plt.show()

    # CELL 06 — Extract grayscale once — automatically reuse saved results
    class GrayscaleImages(Dataset):
        def __len__(self): return len(clean_images)
        def __getitem__(self, index):
            return TF.rgb_to_grayscale(clean_images[index], num_output_channels=3)

    gray_packages = {}
    for name in DISPLAY_NAMES:
        path = OUTPUT_DIR / f'grayscale_{name}.pt'
        if path.exists():
            package = verify_regular(torch.load(path, map_location='cpu', weights_only=True))
            assert package['condition'] == 'grayscale'
            gray_packages[name] = package
            print(name, 'grayscale restored.')
            continue
        backbone, normalize = load_backbone(name)
        features = encode_images(backbone, normalize, GrayscaleImages(), name)
        with torch.no_grad():
            logits = {DISPLAY_NAMES[name]: heads[name](features)}
            if name == 'clip':
                text, scale = clip_text_setup(backbone)
                logits[ZERO_SHOT_NAME] = scale * (features @ text.T)
        package = {'condition':'grayscale', 'baseline_sha256':baseline_hash,
                   'test_indices':torch.as_tensor(test_indices), 'features':features, 'logits':logits}
        atomic_save(package, path)
        gray_packages[name] = package
        del backbone, normalize
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('Grayscale representations ready for all three backbones.')

    # CELL 07 — Load all interventions and validate pairing
    # Each entry stores transformed features, their clean row positions, and logits.
    paired = {name: {} for name in DISPLAY_NAMES}
    all_positions = torch.arange(len(test_indices))
    for name, display_name in DISPLAY_NAMES.items():
        gray = gray_packages[name]
        paired[name]['grayscale'] = {'features':gray['features'], 'positions':all_positions, 'logits':gray['logits']}
        relevant_models = [display_name] + ([ZERO_SHOT_NAME] if name == 'clip' else [])
        paired[name]['cue_conflict'] = {
            'features':cue['features'][name], 'positions':cue_positions,
            'logits':{model:cue['logits'][model] for model in relevant_models}}
        patch = verify_regular(torch.load(TASK_DIR / 'results' / 'patch_shuffle' / f'{name}.pt',
                                          map_location='cpu', weights_only=True))
        paired[name]['patch_shuffle'] = {'features':patch['features'], 'positions':all_positions, 'logits':patch['logits']}
        for displacement in [8, 16, 32]:
            for direction in ['left','right','up','down']:
                path = TASK_DIR / 'results' / 'translation' / f'{name}_{displacement:02d}_{direction}.pt'
                package = verify_regular(torch.load(path, map_location='cpu', weights_only=True))
                assert package['displacement'] == displacement and package['direction'] == direction
                paired[name][f'translation_{displacement:02d}_{direction}'] = {
                    'features':package['features'], 'positions':all_positions, 'logits':package['logits']}
        for condition, data in paired[name].items():
            assert data['features'].shape == clean_features[name][data['positions']].shape
            assert torch.isfinite(data['features']).all()
            assert (data['features'].norm(dim=1) > 0).all(), 'Cosine undefined for a zero vector.'
            for logits in data['logits'].values(): assert logits.shape == (len(data['positions']), num_classes)
    print('All features loaded: grayscale, cue conflict, patch shuffle and 12 translations.')

    stability_pairs, prediction_pairs, stability_summary, prediction_summary, conditional_stability, translation_stability = compute_feature_similarity(paired, clean_features, clean_predictions, test_indices, y_test)

    # CELL 09 — Plot representation stability
    REPRESENTATIVE_TRANSLATION = 'translation_32_right'
    PLOT_CONDITIONS = ['grayscale','cue_conflict',REPRESENTATIVE_TRANSLATION,'patch_shuffle']
    cosine_figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    compact = stability_summary[stability_summary['condition'].isin(PLOT_CONDITIONS)]
    compact.pivot(index='condition', columns='backbone', values='mean_cosine').reindex(PLOT_CONDITIONS).plot.bar(ax=axes[0])
    axes[0].set_ylabel('Mean cosine similarity')
    axes[0].set_title('Representative interventions')
    axes[0].tick_params(axis='x', rotation=20)
    for name, group in translation_stability.groupby('backbone'):
        group = group.sort_values('displacement')
        axes[1].plot([0]+group['displacement'].tolist(), [1.0]+group['mean_cosine'].tolist(), marker='o',label=name)
    axes[1].set_xlabel('Translation (pixels)')
    axes[1].set_ylabel('Mean cosine similarity')
    axes[1].set_title('Translation averaged over four directions')
    axes[1].set_xticks([0,8,16,32])
    axes[1].legend()
    for ax in axes:
        ax.set_ylim(-1,1.02)
        ax.grid(axis='y',alpha=0.2)
    plt.tight_layout()
    plt.show()

    # CELL 10 — Fit ONE joint t-SNE per backbone — cached for reuse
    TSNE_SETTINGS = {'seed':SEED, 'perplexity':30, 'max_iter':1000, 'learning_rate':'auto',
                     'metric':'euclidean', 'init':'pca', 'early_exaggeration':12, 'angle':0.5,
                     'pca_components':50, 'l2_normalize':True, 'conditions':PLOT_CONDITIONS}
    projections = {}
    for name in DISPLAY_NAMES:
        blocks = [clean_features[name]] + [paired[name][condition]['features'] for condition in PLOT_CONDITIONS]
        matrix = F.normalize(torch.cat(blocks).float(), dim=1).numpy()
        labels = np.concatenate([y_test.numpy()] + [y_test[paired[name][c]['positions']].numpy() for c in PLOT_CONDITIONS])
        conditions = np.concatenate([np.repeat('clean',len(test_indices))] + [np.repeat(c,len(paired[name][c]['positions'])) for c in PLOT_CONDITIONS])
        image_ids = np.concatenate([test_indices] + [test_indices[paired[name][c]['positions'].numpy()] for c in PLOT_CONDITIONS])
        fingerprint = hashlib.sha256(matrix.tobytes() + json.dumps(TSNE_SETTINGS,sort_keys=True).encode()).hexdigest()
        coords_path = OUTPUT_DIR / f'tsne_{name}.npz'
        if coords_path.exists():
            with np.load(coords_path, allow_pickle=False) as saved:
                if str(saved['fingerprint'].item()) != fingerprint:
                    raise RuntimeError(f't-SNE inputs/settings changed; use a new output name: {coords_path}')
                coords = saved['coordinates'].copy()
            print(name, 'projection restored.')
        else:
            reduced = PCA(n_components=50, whiten=False, svd_solver='full').fit_transform(matrix)
            coords = TSNE(n_components=2, perplexity=30, max_iter=1000, learning_rate='auto',
                          init='pca', metric='euclidean', early_exaggeration=12,
                          method='barnes_hut', angle=0.5, random_state=SEED, n_jobs=1).fit_transform(reduced)
            np.savez_compressed(coords_path, coordinates=coords, fingerprint=fingerprint)
            print(name, 'joint projection fitted:',len(coords),'points.')
        table = pd.DataFrame({'x':coords[:,0],'y':coords[:,1],'condition':conditions,'label':labels,'official_test_index':image_ids})
        projections[name] = table
        table.to_csv(OUTPUT_DIR / f'tsne_{name}_points.csv',index=False)

    # CELL 11 — Plot classes and conditions in the shared embedding
    projection_figures = {}
    colors = plt.get_cmap('tab10')
    for name, table in projections.items():
        figure, axes = plt.subplots(2, 2, figsize=(14, 11), sharex=True, sharey=True)
        clean_table = table[table['condition']=='clean'].reset_index(drop=True)
        for ax, condition in zip(axes.flat,PLOT_CONDITIONS):
            transformed = table[table['condition']==condition]
            # Only show clean counterparts of the displayed transformed images.
            positions = torch.unique(paired[name][condition]['positions']).numpy()
            selected_clean = clean_table.iloc[positions]
            ax.scatter(selected_clean['x'],selected_clean['y'],c=[colors(int(i)) for i in selected_clean['label']],
                       marker='o',s=22,alpha=0.5,edgecolors='none')
            ax.scatter(transformed['x'],transformed['y'],c=[colors(int(i)) for i in transformed['label']],
                       marker='x',s=25,alpha=0.8,linewidths=0.8)
            ax.set_title(condition.replace('_',' '))
            ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
        handles = [Line2D([],[],marker='o',linestyle='',color=colors(i),label=label) for i,label in enumerate(class_names)]
        handles += [Line2D([],[],marker='o',linestyle='',color='black',label='Clean'),
                    Line2D([],[],marker='x',linestyle='',color='black',label='Transformed')]
        figure.legend(handles=handles,loc='lower center',ncol=6,fontsize=9)
        figure.suptitle(f'{name}: one shared projection; cue-conflict colors = content class')
        figure.tight_layout(rect=(0,0.09,1,0.96))
        projection_figures[name] = figure
        plt.show()

    # CELL 12 — Find representation/prediction mismatches for discussion
    # Rank examples rather than inventing a threshold for a large representation change.
    examples = []
    for model_name, group in prediction_pairs.groupby('model'):
        unchanged = group[group['prediction_unchanged']]
        changed = group[~group['prediction_unchanged']]
        for description, selected in [
            ('Lowest cosine among unchanged predictions', unchanged.nsmallest(3,'cosine_similarity')),
            ('Highest cosine among changed predictions', changed.nlargest(3,'cosine_similarity')),
        ]:
            selected = selected.copy(); selected['example_type'] = description
            examples.append(selected)
    mismatch_examples = pd.concat(examples,ignore_index=True)
    for column in ['clean_prediction','transformed_prediction']:
        mismatch_examples[column + '_class'] = mismatch_examples[column].map(dict(enumerate(class_names)))
    display(mismatch_examples[['model','condition','official_test_index','pair_index','cosine_similarity',
                              'clean_prediction_class','transformed_prediction_class','example_type']])
    print('High cosine can still cross a classifier boundary; unchanged labels can coexist with feature movement.')

    # CELL 13 — SAVE analysis tables, settings and figures
    for name, table in {
        'stability_pairs':stability_pairs, 'stability_summary':stability_summary,
        'prediction_pairs':prediction_pairs, 'prediction_summary':prediction_summary,
        'conditional_stability':conditional_stability, 'translation_stability':translation_stability,
        'mismatch_examples':mismatch_examples,
    }.items(): table.to_csv(OUTPUT_DIR / f'{name}.csv',index=False)
    (OUTPUT_DIR / 'visualization_settings.json').write_text(json.dumps(TSNE_SETTINGS,indent=2),encoding='utf-8')
    cosine_figure.savefig(OUTPUT_DIR / 'cosine_stability.png',dpi=180,bbox_inches='tight')
    for name, figure in projection_figures.items():
        figure.savefig(OUTPUT_DIR / f'tsne_{name}.png',dpi=180,bbox_inches='tight')
    print('Saved analysis to:',OUTPUT_DIR)

